# LLM 9장 따라하기: 소득세법 RAG와 Pinecone

이 노트북은 수업의 `ch9.01 → ch9_06 → ch9_09` 흐름을 한 번에 복습합니다. 원본 노트북은 그대로 두고, 문서 읽기 → 분할 → OpenAI 임베딩 → Pinecone 저장·검색 → LLM 답변을 차례로 실행합니다.

**시작 전**: Jupyter의 Python 커널에서 아래 패키지가 없다면 새 셀에서 `%pip install python-dotenv docx2txt langchain-community langchain-text-splitters langchain-openai langchain-pinecone pinecone`을 실행한 뒤 커널을 다시 시작하세요. OpenAI API 사용에는 계정 요금이 발생할 수 있습니다. 노트북 출력이나 코드에 API 키를 적지 마세요.

**실습 계정**: Pinecone의 `tax-index-openai` 인덱스가 3072차원, cosine, serverless로 준비되어 있어야 합니다. OpenAI의 `text-embedding-3-large` 기본 출력은 3072차원입니다.


## 1. 자료와 키 확인

기존 수업 폴더의 `.env`에서 `OPENAI_API_KEY`를 읽습니다. `PINECONE_API_KEY`가 없으면 실행 중에 숨김 입력으로 받습니다. 입력한 Pinecone 키는 이 노트북 파일에 저장되지 않습니다. 강의에 쓰인 소득세법 문서가 있는지도 확인합니다.


In [1]:
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

COURSE_DIR = Path(r"C:\ai\Source\07_LLM")
DOCX_PATH = COURSE_DIR / "tax_docx" / "소득세법(법률)(제21221호)(20260701).docx"
load_dotenv(COURSE_DIR / ".env")

if not DOCX_PATH.is_file():
    raise FileNotFoundError(f"수업 문서가 없습니다: {DOCX_PATH}")
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("수업 폴더 .env의 OPENAI_API_KEY를 확인하세요.")
if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Pinecone API key: ")
if not os.getenv("PINECONE_API_KEY"):
    raise RuntimeError("PINECONE_API_KEY가 비어 있습니다.")

print("자료와 키 이름을 확인했습니다. 키 값은 출력하지 않습니다.")


Pinecone API key: ········
자료와 키 이름을 확인했습니다. 키 값은 출력하지 않습니다.


## 2. 문서를 작은 조각으로 나누기

RAG는 질문과 가까운 부분을 찾습니다. 따라서 긴 법률 문서를 그대로 한 번에 보내지 않고, 겹치는 부분이 있는 조각으로 나눕니다. 수업 예제와 같이 조각 크기 1500자, 겹침 20자로 설정합니다.


In [2]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=20)
documents = Docx2txtLoader(str(DOCX_PATH)).load_and_split(text_splitter=splitter)
assert documents, "문서에서 텍스트를 읽지 못했습니다."
print(f"문서 조각 수: {len(documents)}")
print(f"첫 조각 예시: {documents[0].page_content[:120]!r}")


C:\Users\water\AppData\Local\Temp\ipykernel_31880\3987768702.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


문서 조각 수: 180
첫 조각 예시: '소득세법\n\n\n\n소득세법\n\n[시행 2026. 7. 1.] [법률 제21221호, 2025. 12. 23., 일부개정]\n\n재정경제부(재산세제과(양도소득세)) 044-215-4312\n\n재정경제부(소득세제과(근로소득)) 0'


## 3. OpenAI 임베딩 차원 확인

임베딩은 글을 숫자 벡터로 바꾸는 단계입니다. 이 호출부터 OpenAI API가 사용됩니다. 반환 길이가 Pinecone 인덱스의 차원과 같아야 저장·검색할 수 있습니다.


In [3]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(model="text-embedding-3-large")
dimension = len(embedding.embed_query("소득세법 실습"))
print(f"임베딩 차원: {dimension}")
assert dimension == 3072, "인덱스와 임베딩 차원이 일치해야 합니다."


임베딩 차원: 3072


## 4. 내 Pinecone 인덱스 확인

Pinecone은 벡터를 저장하고 비슷한 벡터를 빠르게 찾는 서비스입니다. 인덱스는 Pinecone 콘솔에서 이미 만들었습니다. 여기서는 이름, 차원, 거리 기준을 확인합니다. `record count`가 아직 0인 것은 정상입니다.


In [5]:
%pip install -q pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.3 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.20.3 which is incompatible.


In [6]:
from pinecone import Pinecone

INDEX_NAME = "tax-index-openai"
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
if not pc.has_index(INDEX_NAME):
    raise RuntimeError(f"Pinecone 프로젝트에 {INDEX_NAME} 인덱스가 없습니다.")
index_info = pc.describe_index(INDEX_NAME)
print(f"인덱스: {INDEX_NAME}, 차원: {index_info.dimension}, 거리: {index_info.metric}")
assert index_info.dimension == dimension
assert index_info.metric == "cosine"


인덱스: tax-index-openai, 차원: 3072, 거리: cosine


## 5. 처음 한 번 저장하고, 다음부터는 연결하기

기록 수가 0일 때만 수업 문서를 임베딩해 Pinecone에 올립니다. 이미 기록이 있으면 다시 올리지 않고 연결합니다. 중간에 업로드가 끊겨 일부만 저장된 경우에는 무작정 다시 실행하지 말고 기록 수를 확인하세요.


In [7]:
from langchain_pinecone import PineconeVectorStore

index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()
record_count = stats.get("total_vector_count", 0) if isinstance(stats, dict) else stats.total_vector_count
print(f"기존 벡터 기록 수: {record_count}")

if record_count == 0:
    print("처음 저장합니다. 문서 길이에 따라 몇 분 걸릴 수 있습니다.")
    database = PineconeVectorStore.from_documents(
        documents=documents, embedding=embedding, index_name=INDEX_NAME
    )
else:
    database = PineconeVectorStore(embedding=embedding, index_name=INDEX_NAME)
    print("기존 인덱스에 연결했습니다. 중복 저장하지 않았습니다.")


C:\Users\water\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


기존 벡터 기록 수: 0
처음 저장합니다. 문서 길이에 따라 몇 분 걸릴 수 있습니다.


## 6. 질문으로 관련 문장 찾기

같은 임베딩 모델로 질문을 벡터로 바꾼 뒤, 가장 가까운 문서 조각 3개를 가져옵니다. 결과가 비어 있다면 Pinecone의 기록 수와 키가 가리키는 프로젝트를 먼저 확인하세요.


In [8]:
question = "소득세법에서 종합소득에는 어떤 소득이 포함되나요?"
retrieved_docs = database.similarity_search(question, k=3)
if not retrieved_docs:
    raise RuntimeError("검색 결과가 없습니다. 인덱스의 기록 수를 확인하세요.")

for number, doc in enumerate(retrieved_docs, start=1):
    print(f"[{number}] {doc.page_content[:300].replace(chr(10), ' ')}")


[1] 제2절 과세표준과 세액의 계산 <개정 2009. 12. 31.>    제1관 세액계산 통칙 <개정 2009. 12. 31.>    제14조(과세표준의 계산) ① 거주자의 종합소득 및 퇴직소득에 대한 과세표준은 각각 구분하여 계산한다.  ② 종합소득에 대한 과세표준(이하 “종합소득과세표준”이라 한다)은 제16조, 제17조, 제19조, 제20조, 제20조의3, 제21조, 제24조부터 제26조까지, 제27조부터 제29조까지, 제31조부터 제35조까지, 제37조, 제39조, 제41조부터 제46조까지, 제46조의2, 제47조 및 제47조의2
[2] 제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 
[3] ③ 연금소득금액은 제1항 각 호에 따른 소득의 금액의 합계액(제2항에 따라 연금소득에서 제외되는 소득과 비과세소득의 금액은 제외하며, 이하 “총연금액”이라 한다)에서 제47조의2에 따른 연금소득공제를 적용한 금액으로 한다.  ④ 연금소득의 범위 및 계산방법과 그 밖에 필요한 사항은 대통령령으로 정한다.  [전문개정 2009. 12. 31.]    제21조(기타소득) ① 기타소득은 이자소득ㆍ배당소득ㆍ사업소득ㆍ근로소득ㆍ연금소득ㆍ퇴직소득 및 양도소득 외의 소득으로서 다음 각 호에서 규정하는 것으로 한다. <개정 2009. 7. 31., 


## 7. 검색 문맥을 LLM에 제공해 답변 받기

검색된 원문이 답변의 근거입니다. 문맥에 답이 없으면 모른다고 답하도록 지시합니다. 이 실습의 답은 학습용이며 실제 세액 계산이나 신고 판단에 사용하지 않습니다.


In [9]:
from langchain_openai import ChatOpenAI

context = "\n\n---\n\n".join(doc.page_content for doc in retrieved_docs)
llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
reply = llm.invoke([
    ("system", "제공된 소득세법 문맥만 근거로 간결하게 답하세요. 근거가 없으면 모른다고 답하세요. 개인별 세액은 계산하지 마세요."),
    ("human", f"질문: {question}\n\n문맥:\n{context}"),
])
print(reply.content)


소득세법에서 종합소득에는 이자소득금액, 배당소득금액, 사업소득금액, 근로소득금액, 연금소득금액, 기타소득금액이 포함됩니다.


## 확인할 것

1. 문서 조각 수가 0보다 큰가?
2. 임베딩 차원이 3072인가?
3. Pinecone 인덱스 이름·차원·거리 기준이 맞는가?
4. 저장 후 Pinecone 콘솔에서 기록 수가 늘었는가?
5. 검색한 원문이 질문과 관계있는가?
6. 답변이 검색 원문으로 설명되는가?

`NameError`는 앞 셀을 건너뛰었을 때 흔합니다. 커널을 다시 시작한 뒤 1번부터 순서대로 실행하세요. `401/403` 오류는 키 또는 프로젝트 권한, `dimension mismatch`는 임베딩 차원과 인덱스 차원 차이를 확인하세요.
